## 1. Setup and Initialization
First, we need to import our libraries and initialize our core components. 

We are building the foundation of a **RAG (Retrieval-Augmented Generation)** system. To do this, we need an **embedding model** (which converts text into numerical vectors that capture meaning) and a **vector database** (which stores and searches those vectors efficiently). We are using `sentence-transformers` for the model and `ChromaDB` for the database.

In [ ]:
from chromadb import Client
from sentence_transformers import SentenceTransformer
import time

# Initialize the embedding model (converts text to vectors)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB and create a collection (like a table in a relational database)
chroma = Client()
collection = chroma.create_collection(name="knowledge_base")

## 2. Preparing the Knowledge Base
A vector database doesn't just store text; it stores **metadata** alongside it. Metadata is crucial because it allows us to filter our searches later (e.g., "only search documents belonging to the Legal department"). 

Here, we are defining our raw documents, attaching rich metadata (category, sensitivity, department) to each one, and generating unique IDs.

In [ ]:
documents = [
    "Customer support chatbots use RAG to provide accurate answers from company documentation.",
    "Data privacy compliance requires careful handling of user information in vector databases.",
    "Scaling vector search to millions of documents needs efficient indexing strategies like HNSW.",
    "Embedding models should be fine-tuned on domain-specific data for better retrieval quality.",
    "Hybrid search combines vector similarity with traditional keyword matching for best results.",
    "Vector database sharding distributes data across multiple nodes to handle large datasets."
]

# Rich metadata for filtering and analysis
metadata = [
    {"category": "AI Applications", "sensitivity": "low", "department": "Product"},
    {"category": "Data Governance", "sensitivity": "high", "department": "Legal"},  
    {"category": "Infrastructure", "sensitivity": "medium", "department": "Engineering"},
    {"category": "ML Engineering", "sensitivity": "low", "department": "Data Science"},
    {"category": "Search Technology", "sensitivity": "low", "department": "Engineering"},
    {"category": "Scalability", "sensitivity": "medium", "department": "Engineering"}
]

doc_ids = [f"kb_{i+1:03d}" for i in range(len(documents))]
print(f"✓ Prepared {len(documents)} documents with metadata")

## 3. Embedding and Storage
Now we convert our human-readable text into machine-readable numbers. 

The `model.encode()` function translates each document into a high-dimensional vector (an array of numbers). Documents with similar meanings will have vectors that are closer together in space. We then load the text, vectors, metadata, and IDs into our ChromaDB collection.

In [ ]:
print("\n🧠 Embedding and storage")
start_time = time.time()

# Convert text into numerical vectors
embeddings = model.encode(documents).tolist()

# Store everything in the vector database
collection.add(documents=documents, embeddings=embeddings, metadatas=metadata, ids=doc_ids)

embed_time = time.time() - start_time
print(f"✓ Embedded and stored {len(documents)} docs in {embed_time:.3f}s")

## 4. Intelligent Search (Semantic Search)
Traditional search looks for exact keyword matches. **Semantic search** looks for *meaning*. 

When we query the database, it converts our search text into a vector and finds the documents that are mathematically closest to it. ChromaDB returns a "distance" score; a lower distance means the vectors are closer. We convert this into a "similarity score" (1 - distance) where a higher number means a better match.

In [ ]:
print("\n🔍 Intelligent search and interpretation")

queries = [
    {"text": "How can we scale our search system?", "context": "Engineering planning"},
    {"text": "What privacy concerns should we consider?", "context": "Compliance review"},
    {"text": "How to improve search accuracy?", "context": "Product optimization"}
]

for i, query_info in enumerate(queries, 1):
    print(f"\n--- Query {i}: {query_info['text']} ---")
    print(f"Context: {query_info['context']}")
    
    # Perform search with timing
    search_start = time.time()
    results = collection.query(query_texts=[query_info['text']], n_results=2)
    search_time = time.time() - search_start
    
    print(f"Search time: {search_time:.4f}s")
    print("Top results:")
    
    for j, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
        # Convert distance to a 0-1 similarity score
        relevance_score = 1 - results["distances"][0][j]  
        print(f"  {j+1}. [{meta['category']}] Score: {relevance_score:.3f}")
        print(f"     {doc}")
        print(f"     Department: {meta['department']} | Sensitivity: {meta['sensitivity']}")

## 5. Advanced Feature: Metadata Filtering (RBAC)
In the real world, not every user should see every document. By using the metadata we defined earlier, we can implement **Role-Based Access Control (RBAC)**. 

This function intercepts the search process and applies a pre-filter based on the user's permission level, ensuring standard employees only see "low" sensitivity data, while admins can see everything.

In [ ]:
print("\n🔒 PRIVACY: Filtering by sensitivity level")

def filter_by_access(query, level="employee"):
    results = collection.query(query_texts=[query], n_results=6)
    
    # Define permission logic
    allowed = ["low"] if level == "employee" else ["low", "medium", "high"]
    
    # Filter results based on metadata
    filtered = [(doc, meta) for doc, meta in zip(results["documents"][0], results["metadatas"][0]) 
                if meta["sensitivity"] in allowed]
    
    print(f"  {level.capitalize()} sees {len(filtered)}/6 results (sensitivity filter applied)")
    return filtered

employee_results = filter_by_access("privacy data", "employee")
admin_results = filter_by_access("privacy data", "admin")

## 6. Advanced Feature: Database Sharding
When a knowledge base grows to millions of documents, a single database node might struggle. **Sharding** solves this by horizontally partitioning (splitting) the data across multiple collections or servers.

Here, we split our documents into two shards. To search, we have to query both shards independently, combine the results, and sort them by the best similarity score. This is known as a **scatter-gather** search pattern.

In [ ]:
print(f"\n🔄 SHARDING: Split data across collections for scale")

# Create 2 distinct shards
shard1 = chroma.create_collection(name="shard_1")
shard2 = chroma.create_collection(name="shard_2")

# Split documents in half
mid = len(documents) // 2
shard1.add(documents=documents[:mid], embeddings=embeddings[:mid], 
           metadatas=metadata[:mid], ids=[f"s1_{i}" for i in range(mid)])
shard2.add(documents=documents[mid:], embeddings=embeddings[mid:], 
           metadatas=metadata[mid:], ids=[f"s2_{i}" for i in range(len(documents)-mid)])
print(f"  ✓ Split {len(documents)} docs: {mid} in shard1, {len(documents)-mid} in shard2")

def search_shards(query):
    # 1. Scatter: Search each shard separately
    results1 = shard1.query(query_texts=[query], n_results=2)
    results2 = shard2.query(query_texts=[query], n_results=2)
    
    # 2. Gather: Combine results from both shards
    all_results = []
    for doc, dist in zip(results1["documents"][0], results1["distances"][0]):
        all_results.append({"doc": doc, "distance": dist, "shard": "shard1"})
    for doc, dist in zip(results2["documents"][0], results2["distances"][0]):
        all_results.append({"doc": doc, "distance": dist, "shard": "shard2"})
    
    # 3. Sort: Rank by best match (lowest distance)
    all_results.sort(key=lambda x: x["distance"])
    return all_results[:2]  # Return top 2 overall

# Test sharded search
sharded_results = search_shards("scaling systems")
print(f"  Sharded search results:")
for i, result in enumerate(sharded_results, 1):
    score = 1 - result['distance']  # Convert distance to similarity score
    print(f"    {i}. [{result['shard']}] Score: {score:.3f} - {result['doc'][:50]}...")